# 02 短期负荷预测建模（成员 B）

本 notebook 围绕四个可检验问题展开：

1. 用时刻 **t 已经可获得的信息预测 t+1** 的下一条 10 分钟三区总负荷，服务于短期调度。
2. 比较“时间 + 气象”与“时间 + 气象 + 历史负荷”，量化历史负荷自相关的增量。
3. 在同一特征组、同一时间切分、同一测试集上比较线性回归、随机森林、梯度提升，并加入“下一步等于当前值”的持久性基线。
4. 所有指标使用完整的 2017-11-01 至 2017-12-30 测试集；图 10 只截取三天，是为了看清 10 分钟曲线，不是把三天当成测试集。

模型选择对应三种建模假设：线性可解释关系、非线性与交互关系、逐步残差修正。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Windows 下显示中文与负号
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('pandas', pd.__version__, '| sklearn', __import__('sklearn').__version__)

## 1. 读入数据与目标设定

沿用成员 A 的清洗口径：列名规整、显式解析月/日/年格式、设为时间索引并升序排列。目标是三区总负荷 `Z_total = Z1 + Z2 + Z3`，因为题目中的应用是全市电网调度；三区合计也比单个分区更平滑。

这里把目标向前移动一条：当前行的特征属于 `t`，`target` 属于 `t+1`。这样“one-step-ahead”在代码中有明确的时间对齐，不会把未来负荷混入特征。

In [ ]:
df = pd.read_csv('data/题4_城市电力负荷.csv', encoding='utf-8-sig')
df.columns = ['DateTime', 'T', 'RH', 'WS', 'GDF', 'DF', 'Z1', 'Z2', 'Z3']
df['DateTime'] = pd.to_datetime(df['DateTime'], format='%m/%d/%Y %H:%M')
df = df.set_index('DateTime').sort_index()
df['Z_total'] = df[['Z1', 'Z2', 'Z3']].sum(axis=1)
assert df.index.is_monotonic_increasing and not df.index.duplicated().any()
assert df.isna().sum().sum() == 0
print('样本数:', len(df))
print('时间范围:', df.index.min(), '->', df.index.max())
print('总负荷全年均值: %.1f kW' % df['Z_total'].mean())

## 2. 特征工程：把“时间规律”和“历史惯性”拆开

- **时间特征**：小时、星期、月份的周期编码和周末标记，表达晚高峰、周周期与季节变化。
- **气象特征**：温度、湿度、风速、两类辐射量，检验外生因素能解释多少负荷变化。
- **历史负荷特征**：当前负荷 `load_now`、10 分钟前 `lag_1`、1 小时前 `lag_6`、1 天前 `lag_144`、1 周前 `lag_1008`。144 和 1008 的选择由图 9 的自相关峰支撑。

两组特征构成消融实验：如果加入历史负荷后误差大幅下降，说明短期预测的主要信息来自负荷惯性；如果无历史负荷时树模型优于线性模型，说明气象与时间的作用存在非线性或交互。

In [ ]:
df['target'] = df['Z_total'].shift(-1)
df['target_time'] = df.index.to_series().shift(-1)
df['hour'] = df.index.hour
df['dow'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df['dow'] >= 5).astype(int)
for name, period in [('hour', 24), ('dow', 7), ('month', 12)]:
    df[f'{name}_sin'] = np.sin(2 * np.pi * df[name] / period)
    df[f'{name}_cos'] = np.cos(2 * np.pi * df[name] / period)
for lag in [0, 1, 6, 144, 1008]:
    df[f'lag_{lag}'] = df['Z_total'].shift(lag)
df['load_now'] = df['lag_0']
WEATHER = ['T', 'RH', 'WS', 'GDF', 'DF']
TIME = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
HISTORY = ['load_now', 'lag_1', 'lag_6', 'lag_144', 'lag_1008']
FEAT_NO_LAG = WEATHER + TIME
FEAT_WITH_LAG = WEATHER + TIME + HISTORY
print('仅气象+时间:', len(FEAT_NO_LAG), '个特征')
print('气象+时间+历史负荷:', len(FEAT_WITH_LAG), '个特征')

## 3. 时间顺序切分与持久性基线

测试集固定为 **2017-11-01 00:00 至 2017-12-30 23:40**，共 8,639 个可预测时刻，覆盖完整的最后两个月。训练集是更早的记录；最开始 7 天因为没有一周前的历史负荷而无法形成完整特征。

训练集还排除了最后一行，因为它的 `target` 已经落到 11 月 1 日，不能让训练标签跨过测试边界。持久性基线 `prediction = load_now` 不需要训练，用来检验复杂模型是否真的超过“下一步≈当前值”。

In [ ]:
SPLIT = pd.Timestamp('2017-11-01')
required = ['target', 'load_now', 'lag_1', 'lag_6', 'lag_144', 'lag_1008']
train = df[(df.index < SPLIT) & (df['target_time'] < SPLIT)].dropna(subset=required).copy()
test = df[df.index >= SPLIT].dropna(subset=required).copy()
y_train = train['target'].values
y_test = test['target'].values
pred_persistence = test['load_now'].values
print(f'训练集: {train.index.min()} -> {train.index.max()}，{len(train):,} 行')
print(f'测试集: {test.index.min()} -> {test.index.max()}，{len(test):,} 行')
print('测试期目标均值: %.1f kW' % test['target'].mean())

## 4. 模型与评估

- **多元线性回归**：可解释的基准下限；若它已经接近其他模型，说明主要关系可以用加法形式表达。
- **随机森林**：捕捉阈值、非线性和特征交互，适合气象变量与负荷关系不规则的情形。
- **梯度提升**：逐步拟合前一步残差，检验残差修正是否带来额外收益。使用 sklearn 内置实现只是工程便利，不能作为选择模型的理由。

三种模型使用同一训练集、同一测试集、同一指标。指标还与持久性基线比较。

In [ ]:
def fit_predict(model, X_train, y_train, X_test, scale=False):
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    model.fit(X_train, y_train)
    return model.predict(X_test), model

def score_row(y_true, y_pred, model_name, feature_name, baseline_rmse):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    return {
        '模型': model_name,
        '特征': feature_name,
        'MSE (kW²)': mse,
        'RMSE (kW)': rmse,
        'MAE (kW)': mean_absolute_error(y_true, y_pred),
        'R²': r2_score(y_true, y_pred),
        '相对RMSE (%)': rmse / np.mean(y_true) * 100,
        '相对持久性改进 (%)': (1 - (rmse / baseline_rmse) ** 2) * 100,
    }

persistence_rmse = np.sqrt(mean_squared_error(y_test, pred_persistence))
rows = [score_row(y_test, pred_persistence, '持久性基线', '当前负荷', persistence_rmse)]
pred_store = {('持久性基线', '当前负荷'): pred_persistence}
models = {
    '线性回归': (LinearRegression(), True),
    '随机森林': (RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_leaf=2, n_jobs=1, random_state=42), False),
    '梯度提升': (HistGradientBoostingRegressor(random_state=42), False),
}
model_store = {}
for model_name, (model, need_scale) in models.items():
    for feature_name, features in [('仅气象+时间', FEAT_NO_LAG), ('气象+时间+历史负荷', FEAT_WITH_LAG)]:
        pred, fitted = fit_predict(model, train[features], y_train, test[features], scale=need_scale)
        rows.append(score_row(y_test, pred, model_name, feature_name, persistence_rmse))
        pred_store[(model_name, feature_name)] = pred
        model_store[(model_name, feature_name)] = fitted
results = pd.DataFrame(rows)
print(results.to_string(index=False, formatters={
    'MSE (kW²)': '{:,.0f}'.format, 'RMSE (kW)': '{:,.1f}'.format,
    'MAE (kW)': '{:,.1f}'.format, 'R²': '{:.4f}'.format,
    '相对RMSE (%)': '{:.2f}'.format, '相对持久性改进 (%)': '{:.1f}'.format}))

### 如何读评估表

- **MSE/RMSE** 衡量大误差；RMSE 与负荷同单位。
- **MAE** 是典型绝对误差，比 RMSE 少受极端误差影响。
- **R²** 表示测试期波动被解释的比例，但不能单独作为结论。
- **相对 RMSE** 以测试期平均负荷为分母。
- **相对持久性改进** 衡量复杂模型相对最简单基线减少了多少平方误差。

## 5. 图 9：自相关是选择历史特征的依据

ACF 回答“为什么选 1 天、1 周这些滞后”，而不是直接证明模型准确。

In [ ]:
lags = np.arange(0, 2017, 6)
acf = [1.0] + [df['Z_total'].autocorr(int(lag)) for lag in lags[1:]]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lags, acf, color='#1f77b4', lw=1.2)
ax.axhline(0, color='k', lw=0.6)
for lag, label, color in [(144, '1天（144步）', '#d62728'), (1008, '1周（1008步）', '#2ca02c')]:
    ax.axvline(lag, color=color, ls='--', lw=1.2, label=label)
ax.set_xlabel('滞后步数（每步10分钟）'); ax.set_ylabel('自相关系数')
ax.set_title('图9  总负荷自相关：历史负荷为何值得加入')
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('figures/fig_B1_acf.png', dpi=160); plt.show()
print('saved figures/fig_B1_acf.png')

## 6. 图 10：完整测试期 + 三天局部放大

完整测试期的日均曲线放在上方，验证模型没有只在某三天有效；下方两个面板使用同一个 2017-11-20 至 11-22 窗口，比较无历史负荷与有历史负荷。三天只是为了看清 10 分钟曲线，表格与图 12 仍使用全部 8,639 个测试点。

In [ ]:
pred_frame = pd.DataFrame(index=test.index)
pred_frame['真实'] = y_test
pred_frame['持久性基线'] = pred_persistence
for model_name in models:
    pred_frame[model_name] = pred_store[(model_name, '气象+时间+历史负荷')]
daily = pred_frame.resample('D').mean()
window = (test.index >= '2017-11-20') & (test.index < '2017-11-23')
colors = {'线性回归': '#1f77b4', '随机森林': '#ff7f0e', '梯度提升': '#2ca02c'}
fig = plt.figure(figsize=(14, 8)); gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1])
ax0 = fig.add_subplot(gs[0, :])
for col, color in [('真实', 'black'), ('持久性基线', '#888888'), ('线性回归', colors['线性回归']), ('随机森林', colors['随机森林']), ('梯度提升', colors['梯度提升'])]:
    ax0.plot(daily.index, daily[col], lw=1.2 if col == '真实' else 1.0, color=color, label=col)
ax0.set_title('完整测试期（11–12月）日均负荷：定量评估覆盖全部测试点')
ax0.set_ylabel('日均总负荷 (kW)'); ax0.legend(ncol=5, fontsize=8); ax0.grid(alpha=0.3)
for ax, feature_name, title in [(fig.add_subplot(gs[1, 0]), '仅气象+时间', '局部三天：仅气象+时间'), (fig.add_subplot(gs[1, 1]), '气象+时间+历史负荷', '局部三天：加入历史负荷')]:
    ax.plot(test.index[window], y_test[window], color='black', lw=1.3, label='真实')
    for model_name, color in colors.items():
        ax.plot(test.index[window], pred_store[(model_name, feature_name)][window], color=color, lw=0.9, label=model_name)
    ax.set_title(title); ax.set_xlabel('时间'); ax.grid(alpha=0.3); ax.tick_params(axis='x', rotation=25)
fig.axes[1].set_ylabel('总负荷 (kW)'); fig.axes[1].legend(fontsize=8); fig.axes[2].legend(fontsize=8)
fig.suptitle('图10  测试期覆盖与局部预测曲线', y=1.01); fig.tight_layout()
fig.savefig('figures/fig_B2_pred.png', dpi=160, bbox_inches='tight'); plt.show()
print('saved figures/fig_B2_pred.png')

## 7. 图 11：诊断“准不准”与“有没有系统偏差”

下面用测试集 RMSE 最低的配置诊断。散点看贴近程度，残差分布看整体偏差，按小时箱线图看晚高峰是否有系统性高估或低估。

In [ ]:
best_row = results[results['模型'] != '持久性基线'].sort_values('RMSE (kW)').iloc[0]
best_key = (best_row['模型'], best_row['特征']); best_pred = pred_store[best_key]
resid = y_test - best_pred
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].scatter(y_test, best_pred, s=5, alpha=0.25, color='#1f77b4')
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
axes[0].plot(lims, lims, 'k--', lw=1); axes[0].set_xlabel('真实负荷 (kW)'); axes[0].set_ylabel('预测负荷 (kW)')
axes[0].set_title(f'预测 vs 真实\n{best_row["模型"]}·{best_row["特征"]}')
axes[1].hist(resid, bins=60, color='#ff7f0e', alpha=0.8); axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_xlabel('残差 = 真实 − 预测 (kW)'); axes[1].set_title('残差分布')
hour_resid = pd.DataFrame({'hour': test.index.hour, 'resid': resid})
axes[2].boxplot([hour_resid.loc[hour_resid['hour'] == h, 'resid'] for h in range(24)], tick_labels=[str(h) for h in range(24)], showfliers=False)
axes[2].axhline(0, color='k', lw=0.8); axes[2].set_xlabel('预测时刻（小时）'); axes[2].set_ylabel('残差 (kW)'); axes[2].set_title('按小时的残差')
for ax in axes: ax.grid(alpha=0.25)
fig.suptitle('图11  最优配置的误差诊断', y=1.02); fig.tight_layout()
fig.savefig('figures/fig_B3_scatter_resid.png', dpi=160, bbox_inches='tight'); plt.show()
print('诊断配置:', best_row['模型'], '+', best_row['特征']); print('测试集平均残差: %.1f kW' % resid.mean()); print('saved figures/fig_B3_scatter_resid.png')

## 8. 图 12：把“误差水平”和“历史特征收益”分开比较

左侧看 RMSE、MAE 和持久性基线，回答“误差有多大”；右侧看同一模型加入历史负荷后 RMSE 下降比例，回答“信息来源带来多少收益”。这样不会让仅气象模型相对持久性基线的巨大负改进值把图形压扁。

In [ ]:
plot_res = results[results['模型'] != '持久性基线'].copy()
labels = [f'{m}\n{f.replace("气象+时间+", "+")}' for m, f in zip(plot_res['模型'], plot_res['特征'])]
x = np.arange(len(plot_res)); fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
bar_colors = ['#9ecae1' if f == '仅气象+时间' else '#3182bd' for f in plot_res['特征']]
axes[0].bar(x - 0.18, plot_res['RMSE (kW)'], width=0.36, color=bar_colors, label='RMSE')
axes[0].bar(x + 0.18, plot_res['MAE (kW)'], width=0.36, color='#fdae6b', label='MAE')
axes[0].axhline(persistence_rmse, color='black', ls='--', lw=1, label='持久性 RMSE')
axes[0].set_ylabel('误差 (kW)'); axes[0].set_title('RMSE / MAE（越低越好）'); axes[0].set_xticks(x, labels, rotation=20); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25, axis='y')
# 同一模型加入历史负荷后的 RMSE 降幅，直接呈现特征消融的收益
no_lag = results[results['特征'] == '仅气象+时间'].set_index('模型')['RMSE (kW)']
with_lag = results[results['特征'] == '气象+时间+历史负荷'].set_index('模型')['RMSE (kW)']
gain = (1 - with_lag / no_lag) * 100
axes[1].bar(np.arange(len(gain)), gain.values, color='#3182bd')
axes[1].set_ylabel('RMSE 下降比例 (%)'); axes[1].set_title('加入历史负荷后的误差下降')
axes[1].set_xticks(np.arange(len(gain)), gain.index); axes[1].set_ylim(0, 100); axes[1].grid(alpha=0.25, axis='y')
for i, v in enumerate(gain.values): axes[1].text(i, v + 2, f'{v:.1f}%', ha='center', fontsize=9)
fig.suptitle('图12  模型与特征消融的统一比较', y=1.02); fig.tight_layout(); fig.savefig('figures/fig_B4_metrics.png', dpi=160, bbox_inches='tight'); plt.show()
print('saved figures/fig_B4_metrics.png')

## 9. 结论（答辩口径）

- 测试集是 11 月至 12 月全部 8,639 个点；图 10 的三天窗口只是局部放大，完整测试期日均曲线和全量指标都在 notebook 中。
- 三个模型覆盖线性、非线性集成、梯度提升三种假设；持久性基线检验模型是否真正创造增量。
- 图 9 为滞后阶数提供数据依据；特征消融量化历史负荷的贡献；图 11 检查系统性偏差；图 12 同时呈现绝对误差与相对基线改进。
- 本实验只评估 10 分钟一步预测。若改为 1 小时或更长 horizon，需要重新构造目标并重新评估，不能把当前 RMSE 直接外推。

### 与评分维度的对应

- **全面性与深度**：明确三区总负荷、10 分钟预测步长、前10个月训练/后2个月测试；用“仅气象+时间 / 加入历史负荷”消融和持久性基线解释信息来源。
- **方法选择与评估**：线性回归、随机森林、梯度提升覆盖三类建模假设；统一时间切分、统一测试集、统一 MSE/RMSE/MAE/R²，并检查时间泄漏。
- **可视化有效性**：图9解释滞后选择，图10验证完整测试期与局部曲线，图11诊断残差与小时偏差，图12呈现误差水平和历史特征收益。
- **结论与独立思考**：历史负荷贡献远大于仅气象特征；本数据上简单线性模型优于默认树模型；残差在负荷转折时段更宽，后续应加入变化率、节假日或更细气象特征。